# Predictive Maintenance & Anomaly Detection (PMAD) for Industrial Equipment
### Exploratory Data Analysis & Machine Learning Modeling Pipeline
**Domain Context:** High-Reliability Manufacturing & Heavy Rotating Machinery (e.g., Solar Industries India Limited)
**Dataset:** UCI AI4I 2020 Predictive Maintenance Dataset

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root in sys.path
sys.path.insert(0, os.path.abspath('..'))

from src.data_loader import load_dataset
from src.preprocessing import engineer_features, prepare_data_splits
from src.models import train_supervised_models
from src.anomaly_detector import IndustrialAnomalyDetector
from src.recommender import generate_maintenance_recommendation

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
print("All modules imported successfully.")

## 1. Dataset Loading & Schema Inspection

In [ ]:
df = load_dataset()
print(f"Dataset Shape: {df.shape}")
df.head()

In [ ]:
print("Class Distribution of Machine Failure:")
print(df['Machine failure'].value_counts())
print(f"\nFailure Prevalence: {df['Machine failure'].mean() * 100:.2f}%")

## 2. Preventing Target Leakage
The dataset provides 5 diagnostic fault columns (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`). In production telemetry, these are post-incident forensic records, NOT available prior to failure. We strictly eliminate these columns along with arbitrary identifiers (`UDI`, `Product ID`).

In [ ]:
df_feat = engineer_features(df)
print("Engineered Features:")
df_feat[['Temp_Difference_K', 'Power_W', 'Overstrain_Index', 'Speed_Torque_Ratio']].head()

## 3. Exploratory Sensor Telemetry Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=df_feat, x="Torque [Nm]", hue="Machine failure", kde=True, ax=axes[0], palette="tab10")
axes[0].set_title("Torque Distribution by Failure Status")

sns.histplot(data=df_feat, x="Tool wear [min]", hue="Machine failure", kde=True, ax=axes[1], palette="tab10")
axes[1].set_title("Tool Wear Distribution by Failure Status")
plt.tight_layout()
plt.show()

## 4. Train Supervised Predictive Models (Stratified 80/20 Split)

In [ ]:
X_train, X_test, y_train, y_test, feat_cols, scaler = prepare_data_splits(df)
results = train_supervised_models(X_train, y_train, X_test, y_test, scaler, feat_cols)

rf_metrics = results['evaluation_report']['random_forest']
print("Random Forest Results:")
print(f"ROC-AUC: {rf_metrics['roc_auc']:.4f}")
print(f"PR-AUC: {rf_metrics['pr_auc']:.4f}")
print(f"Accuracy: {rf_metrics['accuracy']:.4f}")
print(f"Recall: {rf_metrics['recall']:.4f}")
print(f"F1 Score: {rf_metrics['f1_score']:.4f}")

## 5. Unsupervised Anomaly Detection with Isolation Forest

In [ ]:
detector = IndustrialAnomalyDetector(contamination=0.04)
detector.fit(X_train)
raw_scores, score_pct, is_anom = detector.score_samples(X_test)
print(f"Flagged {is_anom.sum()} anomalies in test set ({is_anom.mean()*100:.2f}%)")

## 6. Physics-Informed Prescriptive Maintenance Recommendation

In [ ]:
sample_telemetry = {
    "Type": "M",
    "Air temperature [K]": 301.2,
    "Process temperature [K]": 306.0, # Low delta T (4.8 K)
    "Rotational speed [rpm]": 1280,   # Low speed
    "Torque [Nm]": 45.0,
    "Tool wear [min]": 210,           # High tool wear
}
rec = generate_maintenance_recommendation(sample_telemetry, failure_probability=0.78, anomaly_severity_pct=88.0)
print(f"Status: {rec['status']}")
print(f"Urgency: {rec['urgency']}")
print("Immediate Actions:", rec['immediate_actions'])
print("Scheduled Actions:", rec['scheduled_actions'])